## DETECTAR ESQUEMA

In [1]:
from pyspark.sql.functions import input_file_name, col, lower, size, split

# ============================================
# 📥 1. LEER ARCHIVOS COMO TEXTO (RAW)
# ============================================

df_raw = spark.read \
    .format("text") \
    .load("Files/raw/dane/year=2026")

df_raw = df_raw.withColumn("file_name", input_file_name())

print("✅ Archivos leídos en modo raw")

# ============================================
# 🧠 2. CONTAR COLUMNAS REALES (split por ;)
# ============================================

df_cols = df_raw.withColumn(
    "num_columns",
    size(split(col("value"), ";"))
)

# ============================================
# 🧠 3. CLASIFICAR ARCHIVOS
# ============================================

df_cols = df_cols.withColumn(
    "file_type",
    when(lower(col("file_name")).rlike("no.?ocupados"), "desocupados")
    .when(lower(col("file_name")).rlike("ocupados"), "ocupados")
    .otherwise("otros")
)

# ============================================
# 📊 4. VALIDACIÓN GLOBAL
# ============================================

df_cols.groupBy("file_type", "num_columns") \
    .count() \
    .orderBy("file_type", "num_columns") \
    .show(100, False)

StatementMeta(, , -1, Cancelled, , Cancelled, True)

In [10]:
spark.table("schema_registry").show(truncate=False)

StatementMeta(, 29c96725-c2a3-4ba7-8dcc-7194ac166ac8, 12, Finished, Available, Finished, False)

+-------------+------+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+--------------------------+------------+---------+--------------------------+
|source_system|entity|version|schema_json                                                                                                                                                                                                                                                                             |schema_hash                     |effective_from            |effective_to|is_active|created_at                |
+-------------+------+-------+------------------------------------------------------------------------------------------------------------------------------

## BLOQUE NUEVO: DETECTAR TYPE DRIFT

In [1]:
# ============================================
# 🧱 CONFIG
# ============================================

from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import *
import hashlib
import json

entity = "labor"
source_system = "dane"

bronze_table = "dane_bronze_lh.bronze_dane_clean_v3"

# ============================================
# 📥 1. LEER BRONZE
# ============================================

df = spark.table(bronze_table)

print("✅ Bronze leído")
print("Columnas:", df.columns)

# ============================================
# 🧠 2. NORMALIZAR TIPOS
# ============================================

def normalize_type(t):
    if isinstance(t, IntegerType):
        return "INT"
    elif isinstance(t, LongType):
        return "BIGINT"
    elif isinstance(t, DoubleType):
        return "DOUBLE"
    elif isinstance(t, FloatType):
        return "FLOAT"
    elif isinstance(t, BooleanType):
        return "BOOLEAN"
    elif isinstance(t, DateType):
        return "DATE"
    elif isinstance(t, TimestampType):
        return "TIMESTAMP"
    else:
        return "STRING"

# ============================================
# 📊 3. SCHEMA ACTUAL
# ============================================

current_schema = {
    field.name: normalize_type(field.dataType)
    for field in df.schema.fields
}

schema_json = df.schema.json()
schema_hash = hashlib.md5(schema_json.encode()).hexdigest()

print("Schema hash:", schema_hash)

# ============================================
# 📜 4. SCHEMA ANTERIOR
# ============================================

prev_schema_df = spark.sql(f"""
SELECT schema_json
FROM schema_registry
WHERE entity = '{entity}'
AND is_active = true
LIMIT 1
""")

# ============================================
# 📜 5. DATA CONTRACTS
# ============================================

contract_df = spark.sql(f"""
SELECT source_column, is_critical, allow_type_change
FROM column_mapping
WHERE entity = '{entity}'
AND is_active = true
""")

contract = {
    row["source_column"]: {
        "is_critical": row["is_critical"] if row["is_critical"] is not None else False,
        "allow_type_change": row["allow_type_change"] if row["allow_type_change"] is not None else True
    }
    for row in contract_df.collect()
}

type_drift_detected = False

# ============================================
# 🔍 6. TYPE DRIFT DETECTION + CONTRACT ENFORCEMENT
# ============================================

if prev_schema_df.count() > 0:

    prev_schema_json = prev_schema_df.collect()[0]["schema_json"]
    prev_schema = json.loads(prev_schema_json)

    prev_fields = {
        f["name"]: f["type"]
        for f in prev_schema["fields"]
    }

    for col_name, current_type in current_schema.items():

        if col_name in prev_fields:

            prev_type = prev_fields[col_name]

            if prev_type != current_type:

                print(f"⚠️ TYPE DRIFT DETECTED → {col_name}: {prev_type} → {current_type}")

                type_drift_detected = True

                rules = contract.get(col_name, {
                    "is_critical": False,
                    "allow_type_change": True
                })

                # ============================================
                # 🚨 DATA CONTRACT ENFORCEMENT
                # ============================================

                if rules["is_critical"] and not rules["allow_type_change"]:
                    
                    print(f"❌ CONTRACT VIOLATION → {col_name}")

                    raise Exception(f"""
                    DATA CONTRACT VIOLATION:
                    Column: {col_name}
                    From: {prev_type}
                    To: {current_type}
                    """)

                else:
                    print(f"⚠️ Drift permitido → {col_name}")

                    # ============================================
                    # 🧾 JSON DETALLE
                    # ============================================

                    details = json.dumps({
                        "column": col_name,
                        "from": prev_type,
                        "to": current_type
                    })

                    # ============================================
                    # 🧠 SCHEMA LOG
                    # ============================================

                    log_schema = StructType([
                        StructField("entity", StringType(), True),
                        StructField("detected_at", TimestampType(), True),
                        StructField("drift_type", StringType(), True),
                        StructField("schema_json", StringType(), True),
                        StructField("details_json", StringType(), True)
                    ])

                    log_data = [(
                        entity,
                        None,
                        "TYPE_CHANGE",
                        prev_schema_json,
                        details
                    )]

                    df_log = spark.createDataFrame(log_data, log_schema) \
                        .withColumn("detected_at", current_timestamp())

                    df_log.write.mode("append").saveAsTable("schema_drift_log")

# ============================================
# 🔄 7. REGISTRAR NUEVO SCHEMA
# ============================================

existing = spark.sql(f"""
SELECT *
FROM schema_registry
WHERE entity = '{entity}'
AND schema_hash = '{schema_hash}'
AND is_active = true
""")

if existing.count() == 0:

    print("⚠️ Nuevo schema detectado")

    version_df = spark.sql(f"""
        SELECT COALESCE(MAX(version), 0) + 1 as next_version
        FROM schema_registry
        WHERE entity = '{entity}'
    """)

    next_version = version_df.collect()[0]["next_version"]

    print(f"📌 Nueva versión: {next_version}")

    # desactivar anterior
    spark.sql(f"""
        UPDATE schema_registry
        SET is_active = false,
            effective_to = current_timestamp()
        WHERE entity = '{entity}'
        AND is_active = true
    """)

    # insertar nuevo
    spark.sql(f"""
        INSERT INTO schema_registry
        VALUES (
            '{source_system}',
            '{entity}',
            {next_version},
            '{schema_json}',
            '{schema_hash}',
            current_timestamp(),
            NULL,
            true,
            current_timestamp()
        )
    """)

    print("✅ Schema registrado")

else:
    print("✅ Schema ya existente")

# ============================================
# 🧪 8. VALIDACIÓN
# ============================================

print("===== SCHEMA REGISTRY =====")
spark.table("schema_registry").show(truncate=False)

print("===== DRIFT LOG =====")
spark.table("schema_drift_log").show(truncate=False)

StatementMeta(, ce39e188-cb43-4255-83d9-573fa10cbb6a, 3, Finished, Available, Finished, False)

✅ Bronze leído
Columnas: ['Year', 'Month', 'Geography', 'Metric']
Schema hash: d987492207bbe932087e6a9f61b6e61a
⚠️ TYPE DRIFT DETECTED → Year: long → BIGINT
❌ CONTRACT VIOLATION → Year


Exception: 
                    DATA CONTRACT VIOLATION:
                    Column: Year
                    From: long
                    To: BIGINT
                    

In [2]:
df = spark.read \
    .format("csv")\
    .option("header", True) \
    .option("delimiter", ";") \
    .option("recursiveFileLookup", "true") \
    .load("Files/raw/dane")

from pyspark.sql.functions import input_file_name

df = df.withColumn("file_name", input_file_name())

StatementMeta(, d3276463-a443-4f3f-9768-a7f4f767d830, 4, Finished, Available, Finished, False)

In [3]:
from pyspark.sql.functions import lower, col

df = df.filter(
    lower(col("file_name")).rlike("ocupados|desocupados|no.?ocupados")
)

print("✅ Solo archivos relevantes")

StatementMeta(, d3276463-a443-4f3f-9768-a7f4f767d830, 5, Finished, Available, Finished, False)

✅ Solo archivos relevantes


In [4]:
print("Columnas detectadas:")
print(df.columns)

StatementMeta(, d3276463-a443-4f3f-9768-a7f4f767d830, 6, Finished, Available, Finished, False)

Columnas detectadas:
['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'HOGAR', 'REGIS', 'AREA', 'CLASE', 'P388', 'P6440', 'P6450', 'P6460', 'P6460S1', 'P6400', 'P6410', 'P6410S1', 'P6422', 'P6424S1', 'P6424S2', 'P6424S3', 'P6426', 'P6430S1', 'P6480', 'P6480S1', 'P9440', 'P6500', 'P6510', 'P6510S1', 'P6510S2', 'P6590', 'P6590S1', 'P6600', 'P6600S1', 'P6610', 'P6610S1', 'P6620', 'P6620S1', 'P6585S1', 'P6585S1A1', 'P6585S1A2', 'P6585S2', 'P6585S2A1', 'P6585S2A2', 'P6585S3', 'P6585S3A1', 'P6585S3A2', 'P6585S4', 'P6585S4A1', 'P6585S4A2', 'P6545', 'P6545S1', 'P6545S2', 'P6580', 'P6580S1', 'P6580S2', 'P6630S1', 'P6630S1A1', 'P6630S2', 'P6630S2A1', 'P6630S3', 'P6630S3A1', 'P6630S4', 'P6630S4A1', 'P6630S6', 'P6630S6A1', 'P6640', 'P6640S1', 'P6765', 'P6765S1', 'P6772', 'P6772S1', 'P6773S1', 'P6775', 'P6750', 'P6760', 'P550', 'P6780', 'P6780S1', 'P6790', 'P1800', 'P1800S1', 'P1801S1', 'P1801S2', 'P1801S3', 'P1802', 'P1879', 'P1805', 'P6800', 'P6810', 'P6810S1', 'P6850', 'P6830', 'P6830S1', 'P6870', 'P6880'

In [3]:
from pyspark.sql.functions import col, input_file_name

df_2019_raw = spark.read \
    .format("text") \
    .option("recursiveFileLookup", "true") \
    .load("Files/raw/dane/year=2019")

df_2019_raw = df_2019_raw.withColumn("file_name", input_file_name())

print("Total registros:")
print(df_2019_raw.count())

StatementMeta(, e4f5ffca-a55d-4ecf-baba-85a6d94d0088, 5, Finished, Available, Finished, False)

Total registros:
5168953


In [4]:
df_2019_raw.select("value").show(10, False)

StatementMeta(, e4f5ffca-a55d-4ecf-baba-85a6d94d0088, 6, Finished, Available, Finished, False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
from pyspark.sql.functions import length, regexp_replace

df_2019_raw.select(
    (length(col("value")) - length(regexp_replace(col("value"), ";", ""))).alias("semicolon_count"),
    (length(col("value")) - length(regexp_replace(col("value"), ",", ""))).alias("comma_count")
).groupBy("semicolon_count", "comma_count").count().show()

StatementMeta(, e4f5ffca-a55d-4ecf-baba-85a6d94d0088, 8, Finished, Available, Finished, False)

+---------------+-----------+------+
|semicolon_count|comma_count| count|
+---------------+-----------+------+
|            165|          0|    12|
|            165|          1|305227|
|             37|          0|    12|
|             37|          1|684230|
|             45|          1|565655|
|             45|          0|    12|
|            164|          0|    24|
|            164|          1|199890|
|             29|          0|    12|
|             29|          1|565655|
|             27|          0|    36|
|             27|          1|808326|
|             36|          1|431485|
|             36|          0|    24|
|             59|          0|    12|
|             59|          1|209726|
|             44|          1|361890|
|             44|          0|    24|
|             26|          0|    24|
|             28|          1|361890|
+---------------+-----------+------+
only showing top 20 rows



In [7]:
from pyspark.sql.functions import split, size

df_2019_cols = df_2019_raw.withColumn(
    "num_columns",
    size(split(col("value"), ";"))
)

df_2019_cols.groupBy("num_columns").count().show()

StatementMeta(, e4f5ffca-a55d-4ecf-baba-85a6d94d0088, 9, Finished, Available, Finished, False)

+-----------+------+
|num_columns| count|
+-----------+------+
|        166|305239|
|         38|684242|
|         46|565667|
|        165|199914|
|         30|565667|
|         28|808362|
|         37|431509|
|         45|361914|
|         60|209738|
|         27|415379|
|         29|361914|
|         59|133068|
|         43| 43262|
|         26| 59357|
|         42| 23721|
+-----------+------+



## PRUEBAS INGESTIÓN EN BRONZE

In [2]:
# ============================================
# 🔍 BRONZE INSPECTION — FILE NAMES
# ============================================

from pyspark.sql import functions as F
from pyspark.sql.functions import input_file_name

year = 2021
bronze_path = f"Files/raw/dane/year={year}"

df = spark.read \
    .format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(bronze_path)

# ✅ FIX CLAVE — crear file_name
df = df.withColumn("file_name", input_file_name())

# ============================================
# 📊 LIST FILES
# ============================================

df.select("file_name").distinct().show(truncate=False)

# ============================================
# 📊 COUNT BY FILE
# ============================================

df.groupBy("file_name").count().orderBy("file_name").show(truncate=False)

StatementMeta(, fabca29f-c9ff-4abb-bfae-ced25a50729e, 4, Finished, Available, Finished, False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file_name                                                                                                                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|abfss://f5879c16-6832-4003-90dd-eb2c10497cc0@onelake.dfs.fabric.microsoft.com/ecf22981-1012-4b17-861f-62880aa61238/Files/raw/dane/year=2021/month=06/Cabecera%20-%20Ocupados.csv?version=1777134626511?flength=8327819                                |
|abf